# Plot geometry from Python

Compose Matplotlib maps, traced lines, profiles and Frenet frames from the same
callable. GUI operation belongs in the [desktop guide](../../docs/gui.md);
standalone PyVista controls and PNG export belong in the
[viewer guide](../../docs/viewer.md).

Run top to bottom in a fresh Python kernel after installing `.[examples]` from
    the repository root. No external data, network access, Qt or PyVista is required.
    Stored outputs are cleared; figures and assertions are generated by the cells.
    [Notebook index](README.md) · [Analysis guide](../../docs/geometry_analysis.md)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mageometry import viz, trace_field_lines, GriddedField
from mageometry.geometry import field_line_transverse_geometry

def field(x, y, z):
    x, y, z = np.broadcast_arrays(x, y, z)
    return -y, x, np.ones_like(z)

## Named quantities and a custom transverse diagnostic

Plot functions accept an existing axes. The named `alpha` in the general plotting
registry uses the legacy Frenet current reconstruction; to use first-gradient
transverse alpha (including straight fields), pass an explicit callable.
Custom quantities accept `(field, x, y, z)`; set their units on the plot.

In [ ]:
def transverse_alpha(field, x, y, z):
    return field_line_transverse_geometry(field, x, y, z, delta=.001)['alpha']

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
viz.plot_geometry_map(field, 'curvature', plane='xy', extent=(-2, 2, -2, 2),
                      n=41, delta=.001, ax=axes[0], unit='m')
viz.plot_geometry_map(field, transverse_alpha, plane='xy', extent=(-2, 2, -2, 2),
                      n=41, ax=axes[1], unit='m')
axes[1].set_title('First-gradient alpha [1/m]')
fig.tight_layout()
plt.show()

## Trace once, plot several ways

The tracer returns physical coordinates and signed arc length about the seed.
Changing a colour quantity does not require tracing again. Undefined geometry
is left blank by the plotting helpers.

In [ ]:
trace = trace_field_lines(field, [.5, 1., 1.5], [0., 0., 0.], [0., 0., 0.], direction='both',
                         ds=.05, max_steps=70)
viz.plot_field_lines(trace, plane='xy', field=field, color='curvature', delta=.001, unit='m')
plt.show()
figure = plt.figure()
axes_3d = figure.add_subplot(111, projection='3d')
viz.plot_field_lines(trace, ax=axes_3d, field=field, color='torsion', delta=.001, unit='m')
plt.show()
viz.plot_line_profiles(trace, field, quantities=('curvature', 'torsion'), delta=.001, unit='m')
plt.show()

## Frame arrows and gridded input

Arrows show T, n, b at fixed positions. The same plotting API accepts a
`GriddedField.field()` interpolant. Stay inside its bounds with room for all
stencils, and keep magnetic/coordinate units consistent.

In [ ]:
viz.plot_frenet_frame(field, [.5, 1., 1.5], 0., 0., delta=.001,
                     plane='xy', length=.25, unit='m')
plt.show()
axis = np.linspace(-2., 2., 25)
coords = np.meshgrid(axis, axis, axis, indexing='ij')
grid = GriddedField(axis, axis, axis, *field(*coords),
                    metadata={'length_unit': 'm', 'field_unit': 'T'})
viz.plot_geometry_map(grid.field(), 'curvature', plane='xy', extent=(-1.5, 1.5, -1.5, 1.5),
                      n=31, delta=.05, unit='m')
plt.show()

Figures are returned as Matplotlib objects; save a selected figure explicitly
with `figure.savefig(path)` when needed. This notebook does not write images into
the repository. Publication concept figures and GUI capture scripts live in
`benchmark/`, with provenance in the [image index](../../docs/images/README.md).